# 03 Gold GeoAnalytics -- Vision Zero LA

**Workspace:** CTS-Geo Analytics GBD  
**Lakehouse:** VisionZero_Lakehouse  

**Purpose:** Run advanced spatial analytics on the Silver crash data using
ArcGIS GeoAnalytics for Fabric and H3 aggregation. Produces multiple Gold
Delta tables consumed by the export notebook and the CityPulse web app.

**Analyses:**
1. Hot Spot Analysis (Getis-Ord Gi*)
2. Space-Time Emerging Hot Spots
3. Kernel Density Estimation
4. H3 Heatmap Aggregation (res 8 + 9)
5. Danger Corridors (road-crash matching)
6. Summary Statistics
7. Monthly Time Series

**Input:** `silver_crashes`  
**Outputs:** `gold_hotspots`, `gold_emerging_hotspots`, `gold_crash_density`,
`gold_h3_heatmap`, `gold_danger_corridors`, `gold_stats`, `gold_timeseries`

In [ ]:
%pip install h3

In [ ]:
import json
import math
import h3
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, IntegerType, DoubleType
from geoanalytics_fabric.sql import functions as GA

LAKEHOUSE = "VisionZero_Lakehouse"
OUTPUT_PATH = "/lakehouse/default/Files/citypulse_output"

# ── Read silver_crashes ────────────────────────────────────────────────────────
silver_crashes = spark.read.format("delta").table(f"{LAKEHOUSE}.silver_crashes")
silver_crashes.cache()
total = silver_crashes.count()
print(f"silver_crashes loaded: {total:,} rows")

In [ ]:
# ── Hot Spot Analysis (Getis-Ord Gi*) ────────────────────────────────────────
#
# Identifies statistically significant spatial clusters of high and low
# crash density using hexagonal bins.

hotspots = GA.find_hot_spots(
    input_layer=silver_crashes,
    bin_type="hexagon",
    bin_size=200,
    bin_size_unit="meters",
    neighborhood_size=3
)

hotspot_count = hotspots.count()
print(f"Hot spot bins: {hotspot_count:,}")
hotspots.select("bin_id", "count", "z_score", "p_value", "classification").show(10)

(
    hotspots
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{LAKEHOUSE}.gold_hotspots")
)
print(f"Saved to {LAKEHOUSE}.gold_hotspots")

In [ ]:
# ── Space-Time Emerging Hot Spots ─────────────────────────────────────────────
#
# Detects trends in crash clustering over time -- intensifying, diminishing,
# new, sporadic, etc.

emerging = GA.find_space_time_hot_spots(
    input_layer=silver_crashes,
    time_field="crash_date",
    bin_type="hexagon",
    bin_size=200,
    bin_size_unit="meters",
    time_step_interval=3,
    time_step_interval_unit="months",
    neighborhood_size=3
)

emerging_count = emerging.count()
print(f"Emerging hot spot bins: {emerging_count:,}")
emerging.groupBy("trend").count().orderBy(F.desc("count")).show()

(
    emerging
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{LAKEHOUSE}.gold_emerging_hotspots")
)
print(f"Saved to {LAKEHOUSE}.gold_emerging_hotspots")

In [ ]:
# ── Kernel Density Estimation ─────────────────────────────────────────────────
#
# Produces a continuous density surface of crash concentration.

density = GA.calculate_density(
    input_layer=silver_crashes,
    weight_field=None,
    search_radius=500,
    search_radius_unit="meters",
    output_cell_size=50,
    output_cell_size_unit="meters"
)

density_count = density.count()
print(f"Density grid cells: {density_count:,}")

(
    density
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{LAKEHOUSE}.gold_crash_density")
)
print(f"Saved to {LAKEHOUSE}.gold_crash_density")

In [ ]:
# ── H3 Heatmap Aggregation ───────────────────────────────────────────────────
#
# Aggregate crash statistics into H3 hexagonal cells at resolutions 8 and 9.
# Compute weighted severity score: fatal=5, pedestrian=4, bicycle=4, severe=3, other=1.

@F.udf(StringType())
def h3_to_boundary_json(h3_index):
    """Return the boundary polygon for an H3 cell as a JSON array of [lng, lat] pairs."""
    try:
        boundary = h3.cell_to_boundary(h3_index)
        coords = [[round(lng, 6), round(lat, 6)] for lat, lng in boundary]
        coords.append(coords[0])  # close the polygon ring
        return json.dumps(coords)
    except Exception:
        return None


# Severity weight mapping
severity_weight = (
    F.when(F.col("severity") == "fatal", 5)
    .when(F.col("severity") == "pedestrian", 4)
    .when(F.col("severity") == "bicycle", 4)
    .when(F.col("severity") == "severe", 3)
    .otherwise(1)
)

silver_weighted = silver_crashes.withColumn("sev_weight", severity_weight)


def build_h3_heatmap(df, h3_col, res_label):
    """Aggregate crashes by H3 cell and compute summary metrics."""
    agg = (
        df.groupBy(h3_col)
        .agg(
            F.count("*").alias("crash_count"),
            F.sum(F.when(F.col("severity") == "fatal", 1).otherwise(0)).alias("fatal_count"),
            F.sum(F.when(F.col("severity") == "pedestrian", 1).otherwise(0)).alias("pedestrian_count"),
            F.sum(F.when(F.col("severity") == "bicycle", 1).otherwise(0)).alias("bicycle_count"),
            F.sum(F.col("sev_weight")).alias("severity_score"),
            F.avg("latitude").alias("center_lat"),
            F.avg("longitude").alias("center_lng"),
        )
        .withColumn("boundary", h3_to_boundary_json(F.col(h3_col)))
        .filter(F.col("boundary").isNotNull())
        .withColumnRenamed(h3_col, "h3_index")
        .withColumn("resolution", F.lit(res_label))
    )
    return agg


heatmap_r9 = build_h3_heatmap(silver_weighted, "h3_res9", "9")
heatmap_r8 = build_h3_heatmap(silver_weighted, "h3_res8", "8")
heatmap = heatmap_r9.unionByName(heatmap_r8)

r9_ct = heatmap_r9.count()
r8_ct = heatmap_r8.count()
print(f"H3 heatmap cells: {r9_ct:,} (res 9) + {r8_ct:,} (res 8) = {r9_ct + r8_ct:,}")
heatmap.orderBy(F.desc("crash_count")).show(10)

(
    heatmap
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{LAKEHOUSE}.gold_h3_heatmap")
)
print(f"Saved to {LAKEHOUSE}.gold_h3_heatmap")

In [ ]:
# ── Danger Corridors (road-crash matching) ───────────────────────────────────
#
# Load the LA County road network from Lakehouse Files, build a spatial grid
# index of crashes, match each crash to the nearest road segment within 50 m,
# and color roads green -> yellow -> red by crash density.

import os

road_data_path = f"{OUTPUT_PATH}/la-county.json"

try:
    with open(road_data_path) as f:
        roads = json.load(f)
    print(f"Loaded {len(roads):,} road segments from {road_data_path}")
except FileNotFoundError:
    print(f"Road file not found at {road_data_path}.")
    print("Upload la-county.json to Lakehouse Files/citypulse_output/ first.")
    roads = []


def point_to_segment_dist(px, py, ax, ay, bx, by):
    """Approximate distance from point (px,py) to segment (ax,ay)-(bx,by) in degrees."""
    dx, dy = bx - ax, by - ay
    if dx == 0 and dy == 0:
        return math.sqrt((px - ax) ** 2 + (py - ay) ** 2)
    t = max(0, min(1, ((px - ax) * dx + (py - ay) * dy) / (dx * dx + dy * dy)))
    proj_x, proj_y = ax + t * dx, ay + t * dy
    return math.sqrt((px - proj_x) ** 2 + (py - proj_y) ** 2)


def crash_count_to_color(count, max_count):
    """Map crash count to RGB color: green (safe) -> yellow -> red (dangerous)."""
    if count == 0:
        return [0, 100, 60]
    t = min(1.0, count / max(max_count * 0.3, 1))
    if t < 0.5:
        s = t * 2
        return [int(255 * s), int(200 + 55 * (1 - s)), 0]
    else:
        s = (t - 0.5) * 2
        return [255, int(200 * (1 - s)), 0]


if roads:
    # Build spatial grid index
    GRID_SIZE = 0.005  # ~500 m
    MATCH_THRESHOLD_DEG = 0.0005  # ~55 m

    crash_points = silver_crashes.select("latitude", "longitude", "severity").collect()
    print(f"Building spatial index for {len(crash_points):,} crashes...")

    crash_grid = {}
    for row in crash_points:
        gx = int(row.longitude / GRID_SIZE)
        gy = int(row.latitude / GRID_SIZE)
        key = (gx, gy)
        if key not in crash_grid:
            crash_grid[key] = []
        crash_grid[key].append((row.latitude, row.longitude, row.severity))

    print(f"Grid cells: {len(crash_grid):,}")

    # Match crashes to road segments
    road_crash_density = []
    for road_idx, road in enumerate(roads):
        path = road["path"]
        if len(path) < 2:
            road_crash_density.append({
                "road_index": road_idx, "crash_count": 0,
                "fatal": 0, "pedestrian": 0, "bicycle": 0, "severe": 0
            })
            continue

        road_cells = set()
        for pt in path:
            gx = int(pt[0] / GRID_SIZE)
            gy = int(pt[1] / GRID_SIZE)
            for dx in [-1, 0, 1]:
                for dy in [-1, 0, 1]:
                    road_cells.add((gx + dx, gy + dy))

        candidates = []
        for cell in road_cells:
            if cell in crash_grid:
                candidates.extend(crash_grid[cell])

        if not candidates:
            road_crash_density.append({
                "road_index": road_idx, "crash_count": 0,
                "fatal": 0, "pedestrian": 0, "bicycle": 0, "severe": 0
            })
            continue

        counts = {"total": 0, "fatal": 0, "pedestrian": 0, "bicycle": 0, "severe": 0}
        for clat, clng, sev in candidates:
            min_dist = float("inf")
            for i in range(len(path) - 1):
                d = point_to_segment_dist(
                    clng, clat, path[i][0], path[i][1], path[i + 1][0], path[i + 1][1]
                )
                min_dist = min(min_dist, d)
                if min_dist < MATCH_THRESHOLD_DEG:
                    break
            if min_dist < MATCH_THRESHOLD_DEG:
                counts["total"] += 1
                if sev in counts:
                    counts[sev] += 1

        road_crash_density.append({
            "road_index": road_idx,
            "crash_count": counts["total"],
            "fatal": counts["fatal"],
            "pedestrian": counts["pedestrian"],
            "bicycle": counts["bicycle"],
            "severe": counts["severe"],
        })

        if road_idx % 5000 == 0:
            print(f"  Processed {road_idx:,}/{len(roads):,} roads...")

    max_count = max((r["crash_count"] for r in road_crash_density), default=0) or 1
    print(f"Road-crash matching complete. Max crashes on single road: {max_count}")
    print(f"Roads with crashes: {sum(1 for r in road_crash_density if r['crash_count'] > 0):,}")

    # Build corridors with color and road_name
    corridor_rows = []
    for rd in road_crash_density:
        idx = rd["road_index"]
        road = roads[idx]
        color = crash_count_to_color(rd["crash_count"], max_count)
        corridor_rows.append({
            "road_index": idx,
            "path_json": json.dumps(road["path"]),
            "color_json": json.dumps(color),
            "crash_count": rd["crash_count"],
            "fatal": rd["fatal"],
            "pedestrian": rd["pedestrian"],
            "bicycle": rd["bicycle"],
            "severe": rd["severe"],
            "road_name": road.get("name", ""),
        })

    corridors_df = spark.createDataFrame(corridor_rows)
    (
        corridors_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{LAKEHOUSE}.gold_danger_corridors")
    )
    print(f"Saved {len(corridor_rows):,} corridors to {LAKEHOUSE}.gold_danger_corridors")
else:
    print("Skipped danger corridors -- no road data available.")

In [ ]:
# ── Summary Statistics ────────────────────────────────────────────────────────
#
# Total crashes, fatals, pedestrian, bicycle. Top 20 most dangerous
# intersections. Crash counts by severity. School zone stats.

total_crashes = silver_crashes.count()
fatal_count = silver_crashes.filter(F.col("severity") == "fatal").count()
ped_count = silver_crashes.filter(F.col("severity") == "pedestrian").count()
bike_count = silver_crashes.filter(F.col("severity") == "bicycle").count()
severe_count = silver_crashes.filter(F.col("severity") == "severe").count()
other_count = silver_crashes.filter(F.col("severity") == "other").count()

date_range = silver_crashes.agg(
    F.min("crash_date").alias("min_date"),
    F.max("crash_date").alias("max_date")
).collect()[0]

# Top 20 dangerous intersections
top_intersections = (
    silver_crashes
    .filter(F.col("intersection").isNotNull())
    .groupBy("intersection")
    .agg(
        F.count("*").alias("crash_count"),
        F.sum(F.when(F.col("severity") == "fatal", 1).otherwise(0)).alias("fatal"),
        F.sum(F.when(F.col("severity") == "pedestrian", 1).otherwise(0)).alias("pedestrian"),
        F.sum(F.when(F.col("severity") == "bicycle", 1).otherwise(0)).alias("bicycle"),
        F.avg("latitude").alias("lat"),
        F.avg("longitude").alias("lng"),
    )
    .orderBy(F.desc("crash_count"))
    .limit(20)
    .collect()
)

# School zone stats
try:
    school_zone_total = silver_crashes.filter(F.col("near_school") == 1).count()
    school_zone_fatal = silver_crashes.filter(
        (F.col("near_school") == 1) & (F.col("severity") == "fatal")
    ).count()
except Exception:
    school_zone_total = 0
    school_zone_fatal = 0

# Build stats dict and save as single-row DataFrame
stats_dict = {
    "total_crashes": total_crashes,
    "fatal": fatal_count,
    "pedestrian": ped_count,
    "bicycle": bike_count,
    "severe": severe_count,
    "other": other_count,
    "date_from": str(date_range.min_date),
    "date_to": str(date_range.max_date),
    "school_zone_crashes": school_zone_total,
    "school_zone_fatal": school_zone_fatal,
    "top_intersections_json": json.dumps([
        {
            "name": row.intersection,
            "crashes": row.crash_count,
            "fatal": row.fatal,
            "pedestrian": row.pedestrian,
            "bicycle": row.bicycle,
            "lat": round(row.lat, 5),
            "lng": round(row.lng, 5),
        }
        for row in top_intersections
    ]),
    "by_severity_json": json.dumps({
        "fatal": fatal_count,
        "pedestrian": ped_count,
        "bicycle": bike_count,
        "severe": severe_count,
        "other": other_count,
    }),
}

stats_df = spark.createDataFrame([stats_dict])
(
    stats_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{LAKEHOUSE}.gold_stats")
)

print(f"Saved summary stats to {LAKEHOUSE}.gold_stats")
print(f"  Total crashes  : {total_crashes:,}")
print(f"  Fatal          : {fatal_count:,}")
print(f"  Pedestrian     : {ped_count:,}")
print(f"  Bicycle        : {bike_count:,}")
print(f"  School zone    : {school_zone_total:,} ({school_zone_fatal} fatal)")
print(f"\nTop 5 dangerous intersections:")
for i, row in enumerate(top_intersections[:5]):
    print(f"  {i+1}. {row.intersection}: {row.crash_count} crashes ({row.fatal} fatal)")

In [ ]:
# ── Monthly Time Series ──────────────────────────────────────────────────────

timeseries = (
    silver_crashes
    .groupBy("year_month")
    .agg(
        F.count("*").alias("total"),
        F.sum(F.when(F.col("severity") == "fatal", 1).otherwise(0)).alias("fatal"),
        F.sum(F.when(F.col("severity") == "pedestrian", 1).otherwise(0)).alias("pedestrian"),
        F.sum(F.when(F.col("severity") == "bicycle", 1).otherwise(0)).alias("bicycle"),
        F.sum(F.when(F.col("severity") == "severe", 1).otherwise(0)).alias("severe"),
        F.sum(F.when(F.col("severity") == "other", 1).otherwise(0)).alias("other"),
    )
    .orderBy("year_month")
)

ts_count = timeseries.count()
print(f"Time series months: {ts_count}")
timeseries.show()

(
    timeseries
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{LAKEHOUSE}.gold_timeseries")
)
print(f"Saved to {LAKEHOUSE}.gold_timeseries")

In [ ]:
# ── Gold Layer Summary ────────────────────────────────────────────────────────

gold_tables = [
    "gold_hotspots",
    "gold_emerging_hotspots",
    "gold_crash_density",
    "gold_h3_heatmap",
    "gold_danger_corridors",
    "gold_stats",
    "gold_timeseries",
]

print("=" * 60)
print("Gold Layer Summary")
print("=" * 60)
for table_name in gold_tables:
    try:
        ct = spark.read.format("delta").table(f"{LAKEHOUSE}.{table_name}").count()
        print(f"  {table_name:30s} : {ct:>10,} rows")
    except Exception as e:
        print(f"  {table_name:30s} : ERROR - {e}")
print("=" * 60)
print("Gold layer complete. Proceed to 04_export_viz_json.")